In [6]:
import pandas as pd
import numpy as np

# ----------------------------------------------------
# 1. LOAD CLEAN DATASETS (From Phase 1 / Phase 2)
# ----------------------------------------------------
sub = pd.read_csv("D:/Internship/Streamflix/subscribers.csv")
titles = pd.read_csv("D:/Internship/Streamflix/titles.csv")
wh = pd.read_csv("D:/Internship/Streamflix/watch_history.csv")
wl = pd.read_csv("D:/Internship/Streamflix/watchlist.csv")

# Ensure all date columns are properly formatted to datetime types
wh['watch_date'] = pd.to_datetime(wh['watch_date'])
sub['signup_date'] = pd.to_datetime(sub['signup_date'])
sub['churn_date'] = pd.to_datetime(sub['churn_date'])



In [4]:
print("--- StreamFlix Phase 3: KPI Generation & Insights Engine ---\n")

# ----------------------------------------------------
# 2. CORE BUSINESS KPI CALCULATIONS (10 REQUIRED)
# ----------------------------------------------------
kpis = {}

# KPI 1: Total Watch Hours
kpis['Total Watch Hours'] = wh['watch_duration_min'].sum() / 60


# KPI 2: Active Rate (%)
total_subscribers = len(sub)
active_subscribers = sub['is_active'].sum()
kpis['Active Rate (%)'] = (active_subscribers / total_subscribers) * 100

# KPI 3: Churn Rate (%)
inactive_subscribers = total_subscribers - active_subscribers
kpis['Churn Rate (%)'] = (inactive_subscribers / total_subscribers) * 100

# KPI 4: Average Completion Rate (%)
kpis['Avg Completion Rate (%)'] = wh['completion_pct'].mean()

# KPI 5: Monthly Recurring Revenue (MRR) - Active Subs Only
kpis['Monthly Recurring Revenue ($)'] = sub[sub['is_active'] == True]['monthly_price_usd'].sum()

# KPI 6: Average Revenue Per User (ARPU) - MRR / Active Subs
kpis['ARPU ($)'] = kpis['Monthly Recurring Revenue ($)'] / active_subscribers

# KPI 7: Average Watch Time per Subscriber (Hours / Active Subs)
kpis['Avg Watch Time per Subscriber (Hours)'] = kpis['Total Watch Hours'] / active_subscribers

# KPI 8: Watchlist Conversion Rate (%)
kpis['Watchlist Conversion Rate (%)'] = (wl['watched'].sum() / len(wl)) * 100

# KPI 9: Hit Concentration (%) - Top 10% titles share of total plays
top_10_percent_cutoff = int(np.ceil(0.10 * wh['title_id'].nunique()))
title_counts = wh['title_id'].value_counts()
top_10_percent_plays = title_counts.head(top_10_percent_cutoff).sum()
total_plays = len(wh)
kpis['Hit Concentration (%)'] = (top_10_percent_plays / total_plays) * 100

# KPI 10: Originals Share of Hours (%)
wh_merged_titles = wh.merge(titles[['title_id', 'is_original']], on='title_id', how='left')
originals_watch_min = wh_merged_titles[wh_merged_titles['is_original'] == True]['watch_duration_min'].sum()
total_watch_min = wh['watch_duration_min'].sum()
kpis['Originals Share of Hours (%)'] = (originals_watch_min / total_watch_min) * 100

# Display clearly formatted KPI Board Report
print("=== BOARD REPORT EXECUTIVE KPI DASHBOARD ===")
for kpi_name, value in kpis.items():
    if '$' in kpi_name:
        print(f"{kpi_name:<42} : ${value:,.2f}")
    elif '%' in kpi_name:
        print(f"{kpi_name:<42} : {value:.2f}%")
    else:
        print(f"{kpi_name:<42} : {value:,.2f}")
print("============================================\n")



--- StreamFlix Phase 3: KPI Generation & Insights Engine ---

=== BOARD REPORT EXECUTIVE KPI DASHBOARD ===
Total Watch Hours                          : 3,333,467.73
Active Rate (%)                            : 74.66%
Churn Rate (%)                             : 25.34%
Avg Completion Rate (%)                    : 65.31%
Monthly Recurring Revenue ($)              : $175,868.51
ARPU ($)                                   : $15.70
Avg Watch Time per Subscriber (Hours)      : 297.66
Watchlist Conversion Rate (%)              : 46.23%
Hit Concentration (%)                      : 30.99%
Originals Share of Hours (%)               : 27.17%



In [ ]:
# ----------------------------------------------------
# 3. INTERMEDIATE CHALLENGE: COHORT RETENTION MATRIX (BONUS +5)
# ----------------------------------------------------
print("=== GENERATING COHORT RETENTION MATRIX (BONUS TASK) ===")

# Create a clean cohort grouping baseline via Signup Month
sub['CohortMonth'] = sub['signup_date'].dt.to_period('M')

# To fix the 'inf'/'NaN' issue, base cohort periods on user lifecycle 
#(Signup -> Churn or Last Active)
# Instead of mapping to watch history entries which don't exist for 2016 cohorts
last_active_date = wh.groupby('subscriber_id')['watch_date'].max().to_frame(
    name='LastActiveDate')
cohort_data = sub.merge(last_active_date, on='subscriber_id', how='left')

# Determine real operational activity cutoff per user
cohort_data['ActivityDate'] = pd.to_datetime(
    np.where(
        cohort_data['is_active'] == False, 
        cohort_data['churn_date'], 
        cohort_data['LastActiveDate'].fillna(wh['watch_date'].max())
    )
)

# Fix: Tenure duration calculated directly using subscriber table profiles
cohort_data['CohortPeriod'] = ((cohort_data['ActivityDate'].dt.year -
                                cohort_data['signup_date'].dt.year) * 12 + 
                               (cohort_data['ActivityDate'].dt.month - 
                                cohort_data['signup_date'].dt.month))

# Ensure users who are still active are counted across all intervals up to their current tenure limit
cohort_rows = []
for _, row in cohort_data.iterrows():
    # User was active from month 0 up to their calculated maximum lifetime period month
    for period in range(0, int(row['CohortPeriod']) + 1):
        cohort_rows.append({'CohortMonth': row['CohortMonth'], 'CohortPeriod': period, 'subscriber_id': row['subscriber_id']})

expanded_cohort_df = pd.DataFrame(cohort_rows)

# Create the absolute user size volume pivot table
cohort_pivot = expanded_cohort_df.pivot_table(index='CohortMonth', 
                                              columns='CohortPeriod', 
                                              values='subscriber_id', 
                                              aggfunc='count').fillna(0)

# Calculate retention rates safely against Month 0 baseline volume sizes
cohort_size = cohort_pivot.iloc[:, 0]
retention_matrix = cohort_pivot.divide(cohort_size, axis=0) * 100

# Extract milestones requested by leadership (Month 3, 6, 12)
target_milestones = [3, 6, 12]
retention_milestones = retention_matrix.reindex(columns=target_milestones, fill_value=0.0).round(2)

# Insert true base cohort sizes at start of data grid frame
retention_milestones.insert(0, 'Cohort Size (Month 0)', cohort_size.astype(int))
retention_milestones.columns = ['Cohort Size (Month 0)', 'Month 3 (%)', 'Month 6 (%)', 'Month 12 (%)']

print("\n--- Corrected Summary Sample of Cohort Matrix Milestones ---")
print(retention_milestones.head(12))


=== GENERATING COHORT RETENTION MATRIX (BONUS TASK) ===

--- Corrected Summary Sample of Cohort Matrix Milestones ---
             Cohort Size (Month 0)  Month 3 (%)  Month 6 (%)  Month 12 (%)
CohortMonth                                                               
2016-01                          4       100.00        75.00         75.00
2016-02                          9        88.89        88.89         88.89
2016-03                         12       100.00       100.00        100.00
2016-04                         13       100.00       100.00        100.00
2016-05                         20       100.00        90.00         85.00
2016-06                         21       100.00       100.00         95.24
2016-07                         17       100.00       100.00        100.00
2016-08                         24       100.00       100.00        100.00
2016-09                         21       100.00       100.00        100.00
2016-10                         33        96.97        96

In [7]:

# ----------------------------------------------------
# 4. EXPORT AND SAVE REPORT (CORRECTED SYNTAX MATCH)
# ----------------------------------------------------
# Resolved: Changed the invalid '.to_sheet()' method to the native '.to_excel()'
with pd.ExcelWriter("Phase3_KPI_Calculations_Report.xlsx", engine='openpyxl') as writer:
    # Save the Business KPI Summary Sheet
    pd.DataFrame(list(kpis.items()), columns=['KPI Metric', 'Calculated Value']).to_excel(writer, sheet_name='KPI_Summary', index=False)
    
    # Save the Cohort Retention Milestone Sheet 
    retention_milestones.to_excel(writer, sheet_name='Cohort_Retention')

print("\n[SUCCESS] Phase 3 data cleanly written to 'Phase3_KPI_Calculations_Report.xlsx'")



[SUCCESS] Phase 3 data cleanly written to 'Phase3_KPI_Calculations_Report.xlsx'
